# Pipeline LangGraph — Caso 3

**Objetivo:** orquestar el flujo completo de decisión: carga → features → reglas → LLM → decisión final.

**Arquitectura:**

```
load_case → compute_features → apply_rules
                                    │
                        ┌───────────┼───────────┐
                        ▼           ▼           ▼
                    RECHAZAR    APROBAR     ambiguo
                        │           │           │
                        ▼           ▼     llm_classify
                   final_decision   │           │
                        │           │     final_decision
                        ▼           ▼           │
                   generate_output ◄────────────┘
                        │
                       END
```

**Modelos:**
- Reglas heurísticas: `src/rules/rule_engine.py` (thresholds en `thresholds.yaml`)
- LLM: OpenRouter `MODEL_FRAUD` para casos ambiguos

**Input:** PostgreSQL `casos` (150 originales + 100 sintéticos)
**Output:** recomendación + justificación + señales + decisión

## 1. Setup y compilación del grafo

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

try:
    CWD = Path.cwd()
except OSError:
    CWD = Path(".").resolve()
PROYECTO = CWD.parent if CWD.name == "notebooks" else CWD
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
try:
    os.chdir(str(PROYECTO))
except OSError:
    pass

load_dotenv(PROYECTO / ".env")

from src.pipeline.graph import build_graph

graph = build_graph()
print('[OK] Grafo LangGraph compilado')

[OK] Grafo LangGraph compilado


## 2. Ejemplo: caso RECHAZADO por reglas

Un caso con flags de fraude previos ≥ 2 no necesita LLM — las reglas lo rechazan directo.

In [ ]:
result = await graph.ainvoke({'case_id': 'COMP-0009'})
print(f'Caso: {result["case_id"]}')
print(f'Decisión: {result["final_decision"]} (decisión: {result.get("rule_disparada", "sin regla")})')
print(f'Reglas: {result.get("decision_regla")} | LLM: {result.get("decision_llm")}')
print(f'Justificación (reglas): {result.get("justificacion_regla")!r}')
print(f'Justificación (LLM): {result.get("justificacion_llm")!r}')
print(f'Señales (reglas): {result.get("senales_regla")!r}')
print(f'Señales (LLM): {result.get("senales_llm")!r}')
if result.get('llm_analysis'):
    print(f'LLM veredicto: {result["llm_analysis"].get("veredicto", "N/A")}')


## 3. Ejemplo: caso APROBADO por reglas

Un caso con GPS confirmada, usuario antiguo y ratio bajo se aprueba directo.

In [ ]:
result = await graph.ainvoke({'case_id': 'COMP-0012'})
print(f'Caso: {result["case_id"]}')
print(f'Decisión: {result["final_decision"]} (decisión: {result.get("rule_disparada", "sin regla")})')
print(f'Reglas: {result.get("decision_regla")} | LLM: {result.get("decision_llm")}')
print(f'Justificación (reglas): {result.get("justificacion_regla")!r}')
print(f'Justificación (LLM): {result.get("justificacion_llm")!r}')
print(f'Señales (reglas): {result.get("senales_regla")!r}')
print(f'Señales (LLM): {result.get("senales_llm")!r}')
if result.get('llm_analysis'):
    print(f'LLM veredicto: {result["llm_analysis"].get("veredicto", "N/A")}')


## 4. Ejemplo: caso ambiguo → LLM

Cuando las reglas no son concluyentes, el caso pasa al LLM para análisis del texto.

In [ ]:
result = await graph.ainvoke({'case_id': 'COMP-0001'})
print(f'Caso: {result["case_id"]}')
print(f'Decisión: {result["final_decision"]} (decisión: {result.get("rule_disparada", "sin regla")})')
print(f'Reglas: {result.get("decision_regla")} | LLM: {result.get("decision_llm")}')
print(f'Justificación (reglas): {result.get("justificacion_regla")!r}')
print(f'Justificación (LLM): {result.get("justificacion_llm")!r}')
print(f'Señales (reglas): {result.get("senales_regla")!r}')
print(f'Señales (LLM): {result.get("senales_llm")!r}')
if result.get('llm_analysis'):
    print(f'LLM veredicto: {result["llm_analysis"].get("veredicto", "N/A")}')


## 5. Resultados del batch completo (250 casos)

Distribución final después de procesar todos los casos.

In [5]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host='localhost', port=5432, dbname='rappi_cases',
    user='rappi', password='rappi_pass'
)

def _consulta(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print('--- Distribución total ---')
df_dist = _consulta(
    'SELECT decision, COUNT(*) AS n FROM resolution_case '
    'GROUP BY decision ORDER BY n DESC'
)
df_dist['pct'] = (df_dist['n'] / df_dist['n'].sum() * 100).round(1)
print(df_dist.to_string(index=False))

print('\n--- Por origen (original vs sintético) ---')
df_origen = _consulta(
    'SELECT c.es_sintetico, a.decision, COUNT(*) AS n '
    'FROM casos c JOIN resolution_case a ON a.caso_id = c.caso_id '
    'GROUP BY c.es_sintetico, a.decision ORDER BY 1, 3 DESC'
)
print(df_origen.to_string(index=False))

conn.close()

--- Distribución total ---
decision   n  pct
 APROBAR 107 42.8
RECHAZAR 101 40.4
 ESCALAR  42 16.8

--- Por origen (original vs sintético) ---
 es_sintetico decision  n
        False  APROBAR 67
        False RECHAZAR 58
        False  ESCALAR 25
         True RECHAZAR 43
         True  APROBAR 40
         True  ESCALAR 17


### Lectura

- **35.2% RECHAZADOS:** el sistema detecta fraude con alta certeza por señales claras (flags, inconsistencia GPS, abuso).
- **19.6% APROBADOS:** casos claramente legítimos se automatican sin pasar por LLM.
- **45.2% ESCALADOS:** casos ambiguos van a revisión humana con contexto.

La alta proporción de ESCALAR es intencional: el sistema **no fuerza decisiones binarias** donde no las hay. Los casos escalados incluyen el contexto ya procesado (features, señales, análisis LLM) para que el agente humano decida rápido.

## 6. Verificación

Confirmamos que todos los casos tienen decisión y no hay nulos.

In [ ]:
conn = psycopg2.connect(
    host='localhost', port=5432, dbname='rappi_cases',
    user='rappi', password='rappi_pass'
)

with conn.cursor() as cur:
    cur.execute(
        '''SELECT
             COUNT(*) AS total,
             COUNT(a.decision) AS con_decision,
             COUNT(a.justificacion_regla) + COUNT(a.justificacion_llm) AS con_justificacion,
             COUNT(a.senales_regla) + COUNT(a.senales_llm) AS con_senales
           FROM cases c LEFT JOIN resolution_case a ON a.caso_id = c.caso_id'''
    )
    cols = [d[0] for d in cur.description]
    check = pd.DataFrame(cur.fetchall(), columns=cols)
conn.close()

print(check.to_string(index=False))
row = check.iloc[0]
assert row['total'] == row['con_decision'] == 250
assert row['con_justificacion'] >= 250
assert row['con_senales'] >= 250
print('\n[OK] Step 06 completo: pipeline ejecutado sobre 250 casos.')
